<h1>Bibliotecas</h1>
<i> - Coleta de feed de portais de notícia - necessário para o modelo entender linguagem política<br>
<i> - Coletar diariamente para formar histórico de notícias

In [1]:
import feedparser
import pandas as pd

from datetime import datetime
from pathlib import Path

<h1>Consulta RSS</h1>

<h2>Configuração dos Portais</h2>

In [ ]:
# preferencia_campo define qual campo do feed RSS é usado como resumo:
#   "summary"    — campo summary (padrão; feeds que retornam texto curto)
#   "subtitle"   — campo subtitle do feedparser (G1 tem ~180c aqui; summary tem ~6000c)
#   "title_only" — ignorar resumo (AGENCIA/UOL: summary é artigo completo ou vazio)

feeds_rss = [
    # ── Portais originais ──────────────────────────────────────────────────────
    {
        "portal": "AGENCIA_BRASIL",
        "categoria": "POLITICA",
        "url_feed": "https://agenciabrasil.ebc.com.br/rss/politica/feed.xml",
        "preferencia_campo": "title_only",   # summary é artigo completo (~6000c)
    },
    {
        "portal": "BBC_BRASIL",
        "categoria": "GERAL",
        "url_feed": "https://feeds.bbci.co.uk/portuguese/rss.xml",
        "preferencia_campo": "summary",      # summary ~185c — bom formato
    },
    {
        "portal": "G1_POLITICA",
        "categoria": "POLITICA",
        "url_feed": "https://g1.globo.com/rss/g1/politica/",
        "preferencia_campo": "subtitle",     # subtitle ~180c vs summary ~6000c
    },
    {
        "portal": "UOL_NOTICIAS",
        "categoria": "GERAL",
        "url_feed": "https://rss.uol.com.br/feed/noticias.xml",
        "preferencia_campo": "title_only",   # summary geralmente vazio
    },
    {
        "portal": "PODER360",
        "categoria": "POLITICA",
        "url_feed": "https://www.poder360.com.br/feed/",
        "preferencia_campo": "summary",      # summary ~79c — excelente
    },
    # ── Novos portais ─────────────────────────────────────────────────────────
    {
        "portal": "FOLHA_PODER",
        "categoria": "POLITICA",
        "url_feed": "https://feeds.folha.uol.com.br/poder/rss091.xml",
        "preferencia_campo": "summary",      # summary ~317c — bom formato
    },
    {
        "portal": "VEJA_POLITICA",
        "categoria": "POLITICA",
        "url_feed": "https://veja.abril.com.br/politica/feed/",
        "preferencia_campo": "summary",      # summary ~111c — excelente
    },
    {
        "portal": "METROPOLES",
        "categoria": "POLITICA",
        "url_feed": "https://www.metropoles.com/brasil/politica-brasil/feed",
        "preferencia_campo": "summary",      # summary ~121c — excelente
    },
    {
        "portal": "CORREIO_BRAZILIENSE",
        "categoria": "POLITICA",
        "url_feed": "https://www.correiobraziliense.com.br/politica/rss.xml",
        "preferencia_campo": "summary",      # summary ~147c — bom
    },
    {
        "portal": "CARTACAPITAL",
        "categoria": "POLITICA",
        "url_feed": "https://www.cartacapital.com.br/politica/feed/",
        "preferencia_campo": "summary",      # summary ~80c — excelente
    },
    {
        "portal": "CONGRESSO_EM_FOCO",
        "categoria": "POLITICA",
        "url_feed": "https://congressoemfoco.uol.com.br/feed/",
        "preferencia_campo": "summary",      # summary ~123c — bom
    },
]

<h2>Coleta de Dados</h2>

In [ ]:
def coletar_feed_rss(feed_info):
    portal            = feed_info["portal"]
    categoria         = feed_info["categoria"]
    url_feed          = feed_info["url_feed"]
    preferencia_campo = feed_info.get("preferencia_campo", "summary")

    print(f"\nConsultando portal: {portal}  (campo={preferencia_campo})")

    feed = feedparser.parse(url_feed, request_headers={"User-Agent": "Mozilla/5.0"})

    if feed.bozo:
        print(f"  Aviso: possível problema ao ler o feed de {portal}")

    print(f"  Entradas encontradas: {len(feed.entries)}")

    registros = []
    for noticia in feed.entries:
        if preferencia_campo == "subtitle":
            # G1: subtitle contém o subtítulo real (~180c); summary é o artigo completo
            resumo = noticia.get("subtitle", "")
        elif preferencia_campo == "title_only":
            # AGENCIA_BRASIL / UOL: summary é artigo completo ou vazio
            resumo = ""
        else:
            # "summary" — padrão para feeds que retornam texto curto
            resumo = noticia.get("summary", "")

        registros.append({
            "portal":            portal,
            "categoria":         categoria,
            "preferencia_campo": preferencia_campo,
            "titulo":            noticia.get("title", ""),
            "link":              noticia.get("link", ""),
            "resumo":            resumo,
            "data_publicacao":   noticia.get("published", ""),
            "url_feed":          url_feed,
            "data_coleta":       datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
        })

    return registros


registros = []
for feed_info in feeds_rss:
    registros_feed = coletar_feed_rss(feed_info)
    registros.extend(registros_feed)

df_raw = pd.DataFrame(registros)

print(f"\nTotal consolidado de notícias: {len(df_raw)}")
print(f"\nDistribuição por portal:")
print(df_raw["portal"].value_counts())
print(f"\nDistribuição por preferencia_campo:")
print(df_raw["preferencia_campo"].value_counts())

df_raw.head()

<h2>Extração</h2>

In [4]:
nome_pipeline = "pipeline_noticias_reais"
nome_base = "rss_noticias_reais"
df_exportar = df_raw

data_agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

pasta_raw = Path(f"../dados/{nome_pipeline}/raw")
pasta_raw.mkdir(parents=True, exist_ok=True)

caminho_saida = pasta_raw / f"{nome_base}_raw_{data_agora}.csv"

df_exportar.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print(f"Arquivo bruto salvo em: {caminho_saida}")

print(f"\nTotal de registros extraídos: {len(df_exportar)}")
print(f"\nData e hora da extração: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

Arquivo bruto salvo em: ..\dados\pipeline_noticias_reais\raw\rss_noticias_reais_raw_2026-05-16_14-03-09.csv

Total de registros extraídos: 173

Data e hora da extração: 16/05/2026 14:03:09


In [5]:
df_raw["portal"].value_counts()

portal
G1_POLITICA       100
BBC_BRASIL         38
UOL_NOTICIAS       15
AGENCIA_BRASIL     10
PODER360           10
Name: count, dtype: int64